# We are using incremental flag as an indicator of initial load or incremental load

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("Incremental Flag","0")

In [0]:
increm_flag=dbutils.widgets.get("Incremental Flag")
print(increm_flag)

In [0]:
df_src=spark.sql('''
select DISTINCT mrn,patient_id_hash,first_name,last_name,date_of_birth,gender,race,ethnicity,address_line1,city,state,zip_code,ssn,phone_number,email from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/patient`
''')

In [0]:
if spark.catalog.tableExists('healthcare.gold.dim_patient'):
    df_sink=spark.sql('''
    select * from healthcare.gold.dim_patient
    ''')
else:
    df_sink=spark.sql('''
        select 1 as dim_patient_key,mrn,patient_id_hash,first_name,last_name,date_of_birth,gender,race,ethnicity,address_line1,city,state,zip_code,ssn,phone_number,email,CAST(NULL AS STRING) AS record_hash,
            CAST(NULL AS DATE) AS effective_start_date,
            CAST(NULL AS DATE) AS effective_end_date,
            CAST(NULL AS STRING) AS is_current,
            CAST(NULL AS TIMESTAMP) AS load_timestamp from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/patient`
        where 1=0''')


In [0]:
df_sink.display()

## Generate Hash Value of all the business columns in source

In [0]:
tracked_cols = ["first_name","last_name","date_of_birth","gender","race","ethnicity",
                 "address_line1","city","state","zip_code","ssn","phone_number","email"]

df_src = df_src.withColumn("record_hash",sha2(concat_ws("||", *tracked_cols), 256))

## SCD2
1. Pickup the current entries of flag=Y(active)
2. Left join is performed to pickup all the records from source and matching from target/sink, NULL when no match found for target/sink columns
3. For new records , there won't be any surrogate keys in sink.
4. For exising records, there will be surrogate keys and sink_record_hash. 
Now, we have to check whether record_hash of df_src == sink_record_hash of df_sink. If yes(unchanged entries), then nothing to perform.
If not, then mark the existing record as inactive and update the expiry date to today and and insert the new record with new surrogate key and update the effective date to current date and expiry to high end date.
5. For totally new records, insert the entry with active flag as Yes and update the effective date to 1900-01-01(beginning of the time) and expiry date to high end date.

In [0]:
df_sink_current=df_sink.filter(df_sink["is_current"]=='Y')
df_filter=df_src.join(df_sink_current,df_src["patient_id_hash"]==df_sink_current["patient_id_hash"],"left").select(df_src["*"],df_sink_current["dim_patient_key"],df_sink_current["record_hash"].alias("sink_record_hash"))

In [0]:
df_filter.display()

In [0]:
df_new_record=df_filter.filter(df_filter["dim_patient_key"].isNull())

In [0]:
df_changed_record=df_filter.filter(df_filter["dim_patient_key"].isNotNull() & (df_filter["record_hash"]!=df_filter["sink_record_hash"]))

In [0]:
df_unchanged_record=df_filter.filter(df_filter["record_hash"]==df_filter["sink_record_hash"])

In [0]:
if increm_flag=='0':
    max_value=1
else:
    max_value_df=spark.sql('''
            select coalesce(max(dim_patient_key),0 ) from healthcare.gold.dim_patient
    ''')
    max_value=max_value_df.collect()[0][0]+1

In [0]:
df_insert=df_new_record.unionByName(df_changed_record).withColumn("isNewPatient",col("dim_patient_key").isNull())

In [0]:
df_insert.limit(5).display()

In [0]:
windowSpec=Window.orderBy(col("patient_id_hash"))
df_insert=df_insert.withColumn("dim_patient_key",row_number().over(windowSpec)+lit(max_value-1))\
    .withColumn("effective_start_date",when(col("isNewPatient"),to_date(lit("1900-01-01"))).otherwise(current_date()))\
    .withColumn("effective_end_date",to_date(lit("9999-12-31")))\
    .withColumn("is_current",lit("Y"))\
    .withColumn("load_timestamp",lit(current_timestamp()))

We are dropping sink_record_hash column as it's just a conditional column

In [0]:
df_insert=df_insert.drop("sink_record_hash")

In [0]:
df_insert.limit(10).display()

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("healthcare.gold.dim_patient"):
    deltaTable = DeltaTable.forName(spark, "healthcare.gold.dim_patient")
    #insert new records and new version of old records
    df_insert.write.format("delta")\
        .mode("append")\
            .saveAsTable("healthcare.gold.dim_patient")
    # make the old records inactive
    deltaTable.alias("target").merge(df_changed_record.alias("source"),"target.dim_patient_key=source.dim_patient_key")\
        .whenMatchedUpdate(
            set={
                "effective_end_date":lit(current_date()),
                "is_current": lit("N")
            }
        )\
        .execute()
else:
    df_insert.write.format("delta")\
        .mode("append")\
            .saveAsTable("healthcare.gold.dim_patient")

In [0]:
%sql
select * from healthcare.gold.dim_patient;